<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

## Path Constants

In [ ]:
PROJECT = "cl_st2_ph2_arianne"

In [2]:
from pathlib import Path

SCORES_FILE = Path(f"sas/output_{PROJECT}/{PROJECT}_scores_only.tsv")
MEANS_PATTERN = f"sas/output_{PROJECT}/means_group_f{{dim}}.tsv"
FILE_IDS_PATH = Path("file_ids.txt")

## Load factor scores

In [3]:
import pandas as pd

scores_df = pd.read_csv(SCORES_FILE, sep="\t")

scores_df

,filename,source,model,prompt,group,fac1,fac2,fac3,fac4,v000001,...,v000961,v000962,v000963,v000964,v000965,v000966,v000967,v000968,v000969,v000970
0,t000001,ai,gemini,persona,persona_gemini,7,6,0,2,0,...,1,0,0,0,0,0,0,0,0,0
1,t000002,ai,gemini,persona,persona_gemini,3,7,5,1,0,...,1,0,0,0,0,0,0,0,0,0
2,t000003,ai,gemini,persona,persona_gemini,10,8,0,0,0,...,1,0,0,0,1,0,0,0,0,0
3,t000004,ai,gemini,persona,persona_gemini,4,7,6,0,0,...,1,0,0,0,0,0,0,0,0,0
4,t000005,ai,gemini,persona,persona_gemini,5,7,5,3,0,...,0,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,t003996,human,human,human,human,82,21,8,1,0,...,1,1,1,1,1,0,1,0,0,0
3996,t003997,human,human,human,human,46,12,3,1,0,...,1,1,1,0,1,0,0,0,0,1
3997,t003998,human,human,human,human,39,8,4,3,0,...,1,0,0,0,1,0,0,0,0,0
3998,t003999,human,human,human,human,99,22,21,14,0,...,1,0,1,0,1,0,0,0,0,0


## Load file ID mapping

In [4]:
file_ids_df = pd.read_csv(
    FILE_IDS_PATH,
    sep=" ",
    names=["filename", "subcorpus filename"],
)

file_ids_df.head()

,filename,subcorpus filename
0,t000001,t001_gemini.txt
1,t000002,t002_gemini.txt
2,t000003,t003_gemini.txt
3,t000004,t004_gemini.txt
4,t000005,t005_gemini.txt


## Load group means

In [5]:
import re

factor_cols = [col for col in scores_df.columns if re.fullmatch(r"fac\d+", col)]
factor_cols = sorted(factor_cols, key=lambda col: int(col.replace("fac", "")))

means_dfs = {}

for factor_col in factor_cols:
    fac_num = int(factor_col.replace("fac", ""))
    means_file = Path(MEANS_PATTERN.format(dim=fac_num))

    if not means_file.exists():
        raise FileNotFoundError(f"Means file not found: {means_file}")

    means_df = pd.read_csv(means_file, sep="\t")

    mean_col = f"Mean fac{fac_num}"

    if "group" not in means_df.columns:
        raise ValueError(f"'group' column missing from means file: {means_file}")

    if mean_col not in means_df.columns:
        raise ValueError(f"'{mean_col}' column missing from means file: {means_file}")

    means_df["group"] = means_df["group"].astype(str).str.strip()

    means_dfs[factor_col] = means_df

means_dfs["fac1"].head()

,Effect,group,N,Mean fac1,SD fac1
0,group,human,1000,24.693,14.953540
1,group,persona_gemini,1000,3.998,2.932677
2,group,persona_gpt,1000,12.108,7.512175
3,group,persona_grok,1000,4.569,3.391274


In [6]:
means_dfs

{'fac1':   Effect           group     N  Mean fac1    SD fac1
 0  group           human  1000     24.693  14.953540
 1  group  persona_gemini  1000      3.998   2.932677
 2  group     persona_gpt  1000     12.108   7.512175
 3  group    persona_grok  1000      4.569   3.391274,
 'fac2':   Effect           group     N  Mean fac2    SD fac2
 0  group           human  1000     20.725  11.471660
 1  group  persona_gemini  1000      7.991   4.293874
 2  group     persona_gpt  1000     25.534  12.232354
 3  group    persona_grok  1000      9.008   5.235776,
 'fac3':   Effect           group     N  Mean fac3   SD fac3
 0  group           human  1000     14.831  9.920830
 1  group  persona_gemini  1000      5.007  3.917435
 2  group     persona_gpt  1000     11.757  7.446435
 3  group    persona_grok  1000      5.039  3.955898,
 'fac4':   Effect           group     N  Mean fac4   SD fac4
 0  group           human  1000      8.185  6.647327
 1  group  persona_gemini  1000      2.914  3.321391
 

## Simulate selection process

In [8]:
means_dfs["fac1"].sort_values("Mean fac1", ascending=False)

,Effect,group,N,Mean fac1,SD fac1
0,group,human,1000,24.693,14.953540
2,group,persona_gpt,1000,12.108,7.512175
3,group,persona_grok,1000,4.569,3.391274
1,group,persona_gemini,1000,3.998,2.932677


In [12]:
scores_df.loc[scores_df["group"].eq("human")].nlargest(20, "fac1")

,filename,source,model,prompt,group,fac1,fac2,fac3,fac4,v000001,...,v000961,v000962,v000963,v000964,v000965,v000966,v000967,v000968,v000969,v000970
3451,t003452,human,human,human,human,101,22,14,14,0,...,1,0,1,1,1,0,0,0,0,0
3280,t003281,human,human,human,human,99,27,15,20,0,...,1,1,1,1,1,0,0,0,0,1
3998,t003999,human,human,human,human,99,22,21,14,0,...,1,0,1,0,1,0,0,0,0,0
3022,t003023,human,human,human,human,92,24,8,6,1,...,1,0,1,1,1,0,0,0,0,0
3226,t003227,human,human,human,human,91,51,29,16,0,...,1,0,1,0,1,0,1,0,0,0
3999,t004000,human,human,human,human,91,25,19,7,0,...,0,0,1,1,1,0,1,0,0,1
3272,t003273,human,human,human,human,88,23,10,22,0,...,1,0,1,1,1,0,0,0,0,0
3297,t003298,human,human,human,human,88,42,19,20,0,...,1,0,1,1,1,0,0,0,0,0
3185,t003186,human,human,human,human,85,71,26,12,0,...,1,1,1,0,1,0,0,0,0,0
3300,t003301,human,human,human,human,85,24,12,16,0,...,1,1,1,1,1,0,0,0,0,0


In [11]:
scores_df.loc[scores_df["group"].eq("persona_gpt")].nlargest(10, "fac1")

,filename,source,model,prompt,group,fac1,fac2,fac3,fac4,v000001,...,v000961,v000962,v000963,v000964,v000965,v000966,v000967,v000968,v000969,v000970
1995,t001996,ai,gpt,persona,persona_gpt,56,32,1,4,0,...,1,1,1,0,0,0,0,0,0,1
1228,t001229,ai,gpt,persona,persona_gpt,51,40,7,15,0,...,1,0,1,0,1,0,0,0,0,0
1163,t001164,ai,gpt,persona,persona_gpt,49,40,5,14,0,...,1,0,1,0,1,0,0,0,0,0
1109,t001110,ai,gpt,persona,persona_gpt,46,42,12,7,0,...,1,1,1,0,1,0,1,0,0,0
1099,t001100,ai,gpt,persona,persona_gpt,44,25,3,5,0,...,1,0,1,1,1,0,0,0,0,0
1272,t001273,ai,gpt,persona,persona_gpt,44,32,11,15,0,...,1,0,1,0,1,0,0,0,0,0
1297,t001298,ai,gpt,persona,persona_gpt,44,58,14,10,0,...,1,0,1,0,1,0,0,0,0,0
1994,t001995,ai,gpt,persona,persona_gpt,41,34,2,7,0,...,1,1,1,0,0,0,0,0,0,1
1451,t001452,ai,gpt,persona,persona_gpt,40,31,9,4,1,...,1,0,1,1,1,0,0,0,0,0
1477,t001478,ai,gpt,persona,persona_gpt,38,23,4,2,0,...,1,0,0,0,0,0,0,0,0,0
